In [0]:
# ============================================================
# UK RETAIL COMMERCIAL INTELLIGENCE PLATFORM
# MODULE 2 — TRADE SPEND & PROMOTIONAL ROI
# Author: Byron Kamau
# ============================================================
# DEPENDS ON: Module 7, Module 1
#
# OUTPUTS:
#   gold_promo_events          — promo periods with lift & ROI
#   gold_trade_spend_efficiency — spend per incremental unit
#   gold_promo_roi_ranking     — ranked promo scorecard
# ============================================================

# Module 2 — Trade Spend & Promotional ROI
**Layer:** Silver → Gold

**What this notebook does:**
1. Tags every transaction as promotional or baseline
2. Calculates baseline volume using pre-promo rolling average
3. Calculates promotional lift and incremental revenue
4. Calculates incremental margin from the lift
5. Deducts trade spend cost to calculate true Promotional ROI
6. Ranks all promotions — flags positive, breakeven, value-destroying
7. Builds trade spend efficiency heatmap by retailer

## Cell 1 — Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
import numpy as np

DATABASE = "uk_retail_commercial"
spark.sql(f"USE {DATABASE}")

print("✅ Database set:", DATABASE)

✅ Database set: uk_retail_commercial


## Cell 2 — Load Source Tables

In [0]:
df_silver   = spark.table(f"{DATABASE}.silver_pos_cleaned")
df_promos   = spark.table(f"{DATABASE}.silver_promo_calendar")
df_funding  = spark.table(f"{DATABASE}.silver_retailer_funding")

print(f"silver_pos_cleaned rows : {df_silver.count():,}")
print(f"silver_promo_calendar   : {df_promos.count()} promotions")
print()
print("Promotion Calendar:")
display(df_promos.orderBy("start_date"))

silver_pos_cleaned rows : 958,501
silver_promo_calendar   : 12 promotions

Promotion Calendar:


promo_id,retailer_id,promo_name,start_date,end_date,disc_pct,promo_type,trade_spend_gbp,ingested_at
PRO-001,RET-001,Boots Baby Event,2010-01-15,2010-01-28,20.0,Scan,3000,2026-03-18T10:34:33.742Z
PRO-002,RET-003,Asda Baby Week,2010-02-01,2010-02-07,25.0,Scan,5000,2026-03-18T10:34:33.742Z
PRO-003,RET-002,JL Spring Baby Fair,2010-03-01,2010-03-14,15.0,Feature,2500,2026-03-18T10:34:33.742Z
PRO-004,RET-001,Boots Easter Promo,2010-04-01,2010-04-10,20.0,Scan,2800,2026-03-18T10:34:33.742Z
PRO-005,RET-005,Smyths Summer Sale,2010-06-01,2010-06-30,30.0,Markdown,4000,2026-03-18T10:34:33.742Z
PRO-006,RET-003,Asda Back to School,2010-08-15,2010-08-31,20.0,Scan,4500,2026-03-18T10:34:33.742Z
PRO-007,RET-001,Boots Baby Event Q3,2010-09-01,2010-09-14,20.0,Scan,3200,2026-03-18T10:34:33.742Z
PRO-008,RET-002,JL Christmas Nursery,2010-11-01,2010-11-30,15.0,Feature,3500,2026-03-18T10:34:33.742Z
PRO-009,RET-003,Asda Black Friday Baby,2010-11-26,2010-11-27,35.0,Markdown,6000,2026-03-18T10:34:33.742Z
PRO-010,RET-001,Boots Xmas Event,2010-12-01,2010-12-24,20.0,Scan,4000,2026-03-18T10:34:33.742Z


## Cell 3 — Tag Transactions as Promotional or Baseline
Join each transaction to the promo calendar by retailer and date range

In [0]:
# Add invoice_date as date type for joining
df_silver = df_silver.withColumn("invoice_dt", F.to_date("invoice_date"))

# Cross join transactions with promo calendar then filter by date range
# This tags each transaction with a promo_id if it falls within a promo period
df_tagged = df_silver.join(
    df_promos.select(
        "promo_id", "retailer_id", "promo_name",
        "start_date", "end_date",
        "disc_pct", "promo_type", "trade_spend_gbp"
    ),
    on="retailer_id",
    how="left"
).withColumn(
    "is_promo",
    F.when(
        (F.col("invoice_dt") >= F.col("start_date")) &
        (F.col("invoice_dt") <= F.col("end_date")),
        True
    ).otherwise(False)
)

# Keep only the matching promo tag (where is_promo = True) or no promo
df_promo_tagged = df_tagged.filter(F.col("is_promo") == True)
df_baseline_tagged = df_silver.join(
    df_promo_tagged.select("invoice_no").distinct(),
    on="invoice_no",
    how="left_anti"
).withColumn("is_promo",       F.lit(False)) \
 .withColumn("promo_id",       F.lit(None).cast(StringType())) \
 .withColumn("promo_name",     F.lit("Baseline")) \
 .withColumn("disc_pct",       F.lit(0.0)) \
 .withColumn("promo_type",     F.lit("Baseline")) \
 .withColumn("trade_spend_gbp",F.lit(0)) \
 .withColumn("start_date",     F.lit(None).cast(DateType())) \
 .withColumn("end_date",       F.lit(None).cast(DateType()))

# Union promo and baseline transactions
common_cols = [
    "invoice_no", "stock_code", "description", "category",
    "quantity", "invoice_date", "invoice_dt",
    "invoice_year", "invoice_month", "invoice_quarter",
    "month_label", "unit_price_gbp", "cost_per_unit_gbp",
    "customer_id", "retailer_id", "retailer_name", "channel",
    "uk_region", "gross_revenue_gbp", "net_revenue_gbp",
    "cogs_gbp", "contribution_margin_gbp", "contribution_margin_pct",
    "is_promo", "promo_id", "promo_name",
    "disc_pct", "promo_type", "trade_spend_gbp",
    "start_date", "end_date"
]

df_all_tagged = df_promo_tagged.select(common_cols).unionByName(
    df_baseline_tagged.select(common_cols)
)

promo_rows     = df_all_tagged.filter(F.col("is_promo") == True).count()
baseline_rows  = df_all_tagged.filter(F.col("is_promo") == False).count()

print(f"✅ Promotional transactions : {promo_rows:,}")
print(f"✅ Baseline transactions    : {baseline_rows:,}")

✅ Promotional transactions : 22,874
✅ Baseline transactions    : 935,627


## Cell 4 — Calculate Baseline Volume Per Retailer
Baseline = average weekly revenue in the 4 weeks BEFORE each promotion starts
This is the counterfactual — what would have sold without the promotion

In [0]:
# Add invoice_week from invoice_dt since it's not in df_all_tagged
df_all_tagged = df_all_tagged.withColumn(
    "invoice_week", F.weekofyear("invoice_dt")
).withColumn(
    "invoice_year", F.year("invoice_dt")
)

# Aggregate weekly revenue per retailer (baseline periods only)
df_weekly_baseline = df_all_tagged.filter(F.col("is_promo") == False) \
    .groupBy("retailer_id", "retailer_name", "invoice_year", "invoice_week") \
    .agg(
        F.sum("gross_revenue_gbp").alias("weekly_gross_revenue"),
        F.sum("net_revenue_gbp").alias("weekly_net_revenue"),
        F.sum("contribution_margin_gbp").alias("weekly_contribution_margin"),
        F.sum("quantity").alias("weekly_units")
    )

# Calculate rolling 4-week average baseline per retailer
window_4wk = Window.partitionBy("retailer_id") \
    .orderBy("invoice_year", "invoice_week") \
    .rowsBetween(-4, -1)

df_weekly_baseline = df_weekly_baseline \
    .withColumn("baseline_avg_weekly_revenue",
        F.avg("weekly_gross_revenue").over(window_4wk)) \
    .withColumn("baseline_avg_weekly_units",
        F.avg("weekly_units").over(window_4wk))

print("✅ Baseline weekly averages calculated")
display(df_weekly_baseline.orderBy("retailer_id", "invoice_year", "invoice_week").limit(15))

✅ Baseline weekly averages calculated


retailer_id,retailer_name,invoice_year,invoice_week,weekly_gross_revenue,weekly_net_revenue,weekly_contribution_margin,weekly_units,baseline_avg_weekly_revenue,baseline_avg_weekly_units
RET-001,Boots,2009,49,8869.17999999999,6728.309999999992,3451.1200000000003,4342,null,null
RET-001,Boots,2009,50,7075.539999999988,5489.149999999998,2796.09,3116,8869.17999999999,4342.0
RET-001,Boots,2009,51,5375.539999999996,3988.3399999999983,1974.0299999999982,2887,7972.359999999989,3729.0
RET-001,Boots,2009,52,673.9000000000001,546.2099999999999,308.9,230,7106.753333333324,3448.3333333333335
RET-001,Boots,2010,1,833.4100000000002,440.45,134.20999999999995,956,5498.539999999994,2643.75
RET-001,Boots,2010,2,2972.7499999999995,2051.9600000000005,883.5200000000002,2036,3489.5974999999958,1797.25
RET-001,Boots,2010,4,1545.2200000000003,1131.7600000000007,587.4199999999996,872,2463.899999999999,1527.25
RET-001,Boots,2010,5,1722.790000000002,1178.0400000000004,544.7299999999999,1211,1506.32,1023.5
RET-001,Boots,2010,6,3141.1199999999994,1998.0900000000022,822.8199999999993,2637,1768.5425000000005,1268.75
RET-001,Boots,2010,7,4409.709999999995,3178.4300000000035,1599.5999999999979,2635,2345.4700000000003,1689.0


## Cell 5 — Aggregate Promo Period Performance
Sum actual revenue, units, and margin during each promotion

In [0]:
df_promo_actuals = df_all_tagged.filter(F.col("is_promo") == True) \
    .groupBy(
        "promo_id", "promo_name", "retailer_id", "retailer_name",
        "channel", "promo_type", "disc_pct",
        "start_date", "end_date", "trade_spend_gbp"
    ).agg(
        F.sum("gross_revenue_gbp").alias("actual_gross_revenue"),
        F.sum("net_revenue_gbp").alias("actual_net_revenue"),
        F.sum("contribution_margin_gbp").alias("actual_contribution_margin"),
        F.sum("quantity").alias("actual_units"),
        F.countDistinct("invoice_no").alias("promo_invoices"),
        F.countDistinct("customer_id").alias("promo_customers"),
        F.avg("unit_price_gbp").alias("avg_promo_price"),
        F.datediff(F.col("end_date"), F.col("start_date")).alias("promo_duration_days")
    )

print(f"✅ Promo actuals: {df_promo_actuals.count()} promotion records")
display(df_promo_actuals.orderBy("start_date"))

✅ Promo actuals: 12 promotion records


promo_id,promo_name,retailer_id,retailer_name,channel,promo_type,disc_pct,start_date,end_date,trade_spend_gbp,actual_gross_revenue,actual_net_revenue,actual_contribution_margin,actual_units,promo_invoices,promo_customers,avg_promo_price,promo_duration_days
PRO-001,Boots Baby Event,RET-001,Boots,Retail,Scan,20.0,2010-01-15,2010-01-28,3000,6475.6599999999935,4870.780000000006,2433.51,3288,30,24,3.963461538461541,13
PRO-002,Asda Baby Week,RET-003,Asda,Retail,Scan,25.0,2010-02-01,2010-02-07,5000,14826.920000000026,9512.529999999984,4112.799999999996,8403,40,34,3.0152502553626044,6
PRO-003,JL Spring Baby Fair,RET-002,John Lewis,Retail,Feature,15.0,2010-03-01,2010-03-14,2500,61174.000000000146,46730.81000000018,24368.789999999943,46739,132,106,3.099424284717379,13
PRO-004,Boots Easter Promo,RET-001,Boots,Retail,Scan,20.0,2010-04-01,2010-04-10,2800,2801.77,1527.8199999999997,552.1099999999999,3079,12,10,3.507341772151897,9
PRO-005,Smyths Summer Sale,RET-005,Smyths Toys,Retail,Markdown,30.0,2010-06-01,2010-06-30,4000,50980.77000000007,36340.65999999974,17914.240000000023,29583,138,104,3.1309948006932373,29
PRO-006,Asda Back to School,RET-003,Asda,Retail,Scan,20.0,2010-08-15,2010-08-31,4500,47522.67999999993,30126.69000000008,13123.469999999996,27661,113,92,2.9011799914493124,16
PRO-007,Boots Baby Event Q3,RET-001,Boots,Retail,Scan,20.0,2010-09-01,2010-09-14,3200,10528.240000000016,7603.489999999996,3755.8999999999965,6249,31,22,2.906539440203567,13
PRO-008,JL Christmas Nursery,RET-002,John Lewis,Retail,Feature,15.0,2010-11-01,2010-11-30,3500,173150.3099999997,142112.95999999988,80677.48999999979,92926,400,257,3.094479149140784,29
PRO-009,Asda Black Friday Baby,RET-003,Asda,Retail,Markdown,35.0,2010-11-26,2010-11-27,6000,1741.17,1233.1100000000006,626.5300000000002,755,6,6,3.7793902439024354,1
PRO-010,Boots Xmas Event,RET-001,Boots,Retail,Scan,20.0,2010-12-01,2010-12-24,4000,22819.260000000115,16429.18000000006,8260.240000000025,13689,70,26,2.5791481280520756,23


## Cell 6 — Calculate Baseline for Each Promo Period
Estimate what revenue would have been without the promotion

In [0]:
# Get average baseline revenue per retailer per week
df_retailer_baseline = df_weekly_baseline.groupBy("retailer_id").agg(
    F.avg("weekly_gross_revenue").alias("avg_weekly_baseline_revenue"),
    F.avg("weekly_net_revenue").alias("avg_weekly_baseline_net_revenue"),
    F.avg("weekly_contribution_margin").alias("avg_weekly_baseline_margin"),
    F.avg("weekly_units").alias("avg_weekly_baseline_units")
)

# Join baseline onto promo actuals
df_promo_vs_baseline = df_promo_actuals.join(
    df_retailer_baseline,
    on="retailer_id",
    how="left"
)

# Calculate expected baseline revenue during the promo period
df_promo_vs_baseline = df_promo_vs_baseline \
    .withColumn("promo_weeks",
        F.round((F.col("promo_duration_days") + 1) / 7, 1)) \
    .withColumn("baseline_expected_revenue",
        F.round(F.col("avg_weekly_baseline_revenue") * F.col("promo_weeks"), 2)) \
    .withColumn("baseline_expected_units",
        F.round(F.col("avg_weekly_baseline_units") * F.col("promo_weeks"), 0)) \
    .withColumn("baseline_expected_margin",
        F.round(F.col("avg_weekly_baseline_margin") * F.col("promo_weeks"), 2))

print("✅ Baseline estimates calculated")

✅ Baseline estimates calculated


## Cell 7 — Calculate Promotional Lift & Incrementality

In [0]:
df_lift = df_promo_vs_baseline

# Incremental Volume (Lift)
df_lift = df_lift \
    .withColumn("incremental_units",
        F.round(F.col("actual_units") - F.col("baseline_expected_units"), 0)) \
    .withColumn("incremental_revenue_gbp",
        F.round(F.col("actual_gross_revenue") - F.col("baseline_expected_revenue"), 2)) \
    .withColumn("incremental_margin_gbp",
        F.round(F.col("actual_contribution_margin") - F.col("baseline_expected_margin"), 2))

# Promotional Lift % = (actual - baseline) / baseline
df_lift = df_lift \
    .withColumn("volume_lift_pct",
        F.round(
            F.when(F.col("baseline_expected_units") > 0,
                (F.col("actual_units") - F.col("baseline_expected_units"))
                / F.col("baseline_expected_units") * 100
            ).otherwise(0), 1
        )
    ) \
    .withColumn("revenue_lift_pct",
        F.round(
            F.when(F.col("baseline_expected_revenue") > 0,
                (F.col("actual_gross_revenue") - F.col("baseline_expected_revenue"))
                / F.col("baseline_expected_revenue") * 100
            ).otherwise(0), 1
        )
    )

print("✅ Lift calculations complete")
display(df_lift.select(
    "promo_name", "retailer_name", "promo_type",
    "actual_units", "baseline_expected_units", "incremental_units",
    "volume_lift_pct", "revenue_lift_pct"
).orderBy("start_date"))

✅ Lift calculations complete


promo_name,retailer_name,promo_type,actual_units,baseline_expected_units,incremental_units,volume_lift_pct,revenue_lift_pct
Boots Baby Event,Boots,Scan,3288,8196.0,-4908.0,-59.9,-48.2
Asda Baby Week,Asda,Scan,8403,12935.0,-4532.0,-35.0,-34.9
JL Spring Baby Fair,John Lewis,Feature,46739,33019.0,13720.0,41.6,21.8
Boots Easter Promo,Boots,Scan,3079,5737.0,-2658.0,-46.3,-68.0
Smyths Summer Sale,Smyths Toys,Markdown,29583,28699.0,884.0,3.1,3.7
Asda Back to School,Asda,Scan,27661,31045.0,-3384.0,-10.9,-13.0
Boots Baby Event Q3,Boots,Scan,6249,8196.0,-1947.0,-23.8,-15.8
JL Christmas Nursery,John Lewis,Feature,92926,70992.0,21934.0,30.9,60.4
Asda Black Friday Baby,Asda,Markdown,755,3881.0,-3126.0,-80.5,-74.5
Boots Xmas Event,Boots,Scan,13689,13933.0,-244.0,-1.8,7.4


## Cell 8 — Calculate Promotional ROI
ROI = (Incremental Margin - Trade Spend) / Trade Spend

In [0]:
df_roi = df_lift

# Trade spend is already in the promo calendar
# ROI calculation
df_roi = df_roi \
    .withColumn("trade_spend_gbp", F.col("trade_spend_gbp").cast(DoubleType())) \
    .withColumn("net_incremental_margin_gbp",
        F.round(F.col("incremental_margin_gbp") - F.col("trade_spend_gbp"), 2)) \
    .withColumn("promo_roi_pct",
        F.round(
            F.when(F.col("trade_spend_gbp") > 0,
                F.col("net_incremental_margin_gbp") / F.col("trade_spend_gbp") * 100
            ).otherwise(0), 1
        )
    ) \
    .withColumn("cost_per_incremental_unit_gbp",
        F.round(
            F.when(F.col("incremental_units") > 0,
                F.col("trade_spend_gbp") / F.col("incremental_units")
            ).otherwise(None), 2
        )
    )

# ROI Flag
df_roi = df_roi.withColumn(
    "roi_flag",
    F.when(F.col("promo_roi_pct") >= 20,  "STRONG POSITIVE")
     .when(F.col("promo_roi_pct") >= 0,   "POSITIVE")
     .when(F.col("promo_roi_pct") >= -20, "MARGINAL")
     .otherwise("VALUE DESTROYING")
)

# Incrementality flag
df_roi = df_roi.withColumn(
    "incrementality_flag",
    F.when(F.col("volume_lift_pct") >= 20, "HIGH LIFT")
     .when(F.col("volume_lift_pct") >= 5,  "MODERATE LIFT")
     .when(F.col("volume_lift_pct") >= 0,  "LOW LIFT")
     .otherwise("NO LIFT / CANNIBALISATION")
)

print("✅ ROI calculations complete")

✅ ROI calculations complete


## Cell 9 — Write Gold: Promo Events Table

In [0]:
gold_promo_cols = [
    "promo_id", "promo_name", "retailer_id", "retailer_name",
    "channel", "promo_type", "disc_pct",
    "start_date", "end_date", "promo_duration_days", "promo_weeks",
    "trade_spend_gbp",
    "actual_gross_revenue", "actual_net_revenue",
    "actual_contribution_margin", "actual_units", "promo_invoices",
    "baseline_expected_revenue", "baseline_expected_units",
    "baseline_expected_margin",
    "incremental_units", "incremental_revenue_gbp", "incremental_margin_gbp",
    "volume_lift_pct", "revenue_lift_pct",
    "net_incremental_margin_gbp", "promo_roi_pct",
    "cost_per_incremental_unit_gbp",
    "roi_flag", "incrementality_flag"
]

df_gold_promos = df_roi.select(gold_promo_cols)

(df_gold_promos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_promo_events"))

print(f"✅ gold_promo_events written: {df_gold_promos.count()} promotions")
display(df_gold_promos.orderBy("promo_roi_pct"))

✅ gold_promo_events written: 12 promotions


promo_id,promo_name,retailer_id,retailer_name,channel,promo_type,disc_pct,start_date,end_date,promo_duration_days,promo_weeks,trade_spend_gbp,actual_gross_revenue,actual_net_revenue,actual_contribution_margin,actual_units,promo_invoices,baseline_expected_revenue,baseline_expected_units,baseline_expected_margin,incremental_units,incremental_revenue_gbp,incremental_margin_gbp,volume_lift_pct,revenue_lift_pct,net_incremental_margin_gbp,promo_roi_pct,cost_per_incremental_unit_gbp,roi_flag,incrementality_flag
PRO-011,DTC New Year Sale,RET-006,DTC Ecommerce,DTC,Markdown,25.0,2011-01-01,2011-01-07,6,1.0,1000.0,28615.659999999996,28615.659999999996,18010.540000000005,12555,34,54581.4,20112.0,34652.21,-7557.0,-25965.74,-16641.67,-37.6,-47.6,-17641.67,-1764.2,null,VALUE DESTROYING,NO LIFT / CANNIBALISATION
PRO-004,Boots Easter Promo,RET-001,Boots,Retail,Scan,20.0,2010-04-01,2010-04-10,9,1.4,2800.0,2801.7699999999995,1527.8199999999997,552.1099999999999,3079,12,8751.08,5737.0,3036.56,-2658.0,-5949.31,-2484.45,-46.3,-68.0,-5284.45,-188.7,null,VALUE DESTROYING,NO LIFT / CANNIBALISATION
PRO-001,Boots Baby Event,RET-001,Boots,Retail,Scan,20.0,2010-01-15,2010-01-28,13,2.0,3000.0,6475.660000000001,4870.779999999999,2433.5100000000007,3288,30,12501.54,8196.0,4337.94,-4908.0,-6025.88,-1904.43,-59.9,-48.2,-4904.43,-163.5,null,VALUE DESTROYING,NO LIFT / CANNIBALISATION
PRO-006,Asda Back to School,RET-003,Asda,Retail,Scan,20.0,2010-08-15,2010-08-31,16,2.4,4500.0,47522.680000000015,30126.69000000001,13123.470000000003,27661,113,54624.47,31045.0,15445.55,-3384.0,-7101.79,-2322.08,-10.9,-13.0,-6822.08,-151.6,null,VALUE DESTROYING,NO LIFT / CANNIBALISATION
PRO-002,Asda Baby Week,RET-003,Asda,Retail,Scan,25.0,2010-02-01,2010-02-07,6,1.0,5000.0,14826.920000000002,9512.53,4112.8,8403,40,22760.2,12935.0,6435.65,-4532.0,-7933.28,-2322.85,-35.0,-34.9,-7322.85,-146.5,null,VALUE DESTROYING,NO LIFT / CANNIBALISATION
PRO-009,Asda Black Friday Baby,RET-003,Asda,Retail,Markdown,35.0,2010-11-26,2010-11-27,1,0.3,6000.0,1741.1699999999996,1233.1100000000001,626.53,755,6,6828.06,3881.0,1930.69,-3126.0,-5086.89,-1304.16,-80.5,-74.5,-7304.16,-121.7,null,VALUE DESTROYING,NO LIFT / CANNIBALISATION
PRO-007,Boots Baby Event Q3,RET-001,Boots,Retail,Scan,20.0,2010-09-01,2010-09-14,13,2.0,3200.0,10528.240000000002,7603.489999999999,3755.899999999999,6249,31,12501.54,8196.0,4337.94,-1947.0,-1973.3,-582.04,-23.8,-15.8,-3782.04,-118.2,null,VALUE DESTROYING,NO LIFT / CANNIBALISATION
PRO-012,Amazon Lightning Deal,RET-007,Amazon UK,DTC,Markdown,30.0,2011-02-14,2011-02-14,0,0.1,500.0,2427.6499999999996,2233.41,1430.7600000000002,1443,5,2527.58,1539.0,1432.82,-96.0,-99.93,-2.06,-6.2,-4.0,-502.06,-100.4,null,VALUE DESTROYING,NO LIFT / CANNIBALISATION
PRO-005,Smyths Summer Sale,RET-005,Smyths Toys,Retail,Markdown,30.0,2010-06-01,2010-06-30,29,4.3,4000.0,50980.77000000003,36340.660000000025,17914.24,29583,138,49176.56,28699.0,17261.3,884.0,1804.21,652.94,3.1,3.7,-3347.06,-83.7,4.52,VALUE DESTROYING,LOW LIFT
PRO-010,Boots Xmas Event,RET-001,Boots,Retail,Scan,20.0,2010-12-01,2010-12-24,23,3.4,4000.0,22819.260000000002,16429.179999999997,8260.24,13689,70,21252.61,13933.0,7374.5,-244.0,1566.65,885.74,-1.8,7.4,-3114.26,-77.9,null,VALUE DESTROYING,NO LIFT / CANNIBALISATION


## Cell 10 — Gold: Trade Spend Efficiency Table
Spend per unit of incremental volume by retailer and promo type

In [0]:
gold_efficiency = df_roi.groupBy(
    "retailer_name", "channel", "promo_type"
).agg(
    F.count("promo_id").alias("num_promotions"),
    F.sum("trade_spend_gbp").alias("total_trade_spend_gbp"),
    F.sum("incremental_units").alias("total_incremental_units"),
    F.sum("incremental_revenue_gbp").alias("total_incremental_revenue"),
    F.sum("incremental_margin_gbp").alias("total_incremental_margin"),
    F.avg("promo_roi_pct").alias("avg_roi_pct"),
    F.avg("volume_lift_pct").alias("avg_volume_lift_pct")
).withColumn(
    "total_spend_per_incremental_unit",
    F.round(
        F.when(F.col("total_incremental_units") > 0,
            F.col("total_trade_spend_gbp") / F.col("total_incremental_units")
        ).otherwise(None), 2
    )
).withColumn(
    "spend_efficiency_flag",
    F.when(F.col("total_spend_per_incremental_unit") <= 2.0,  "EFFICIENT")
     .when(F.col("total_spend_per_incremental_unit") <= 5.0,  "MODERATE")
     .otherwise("INEFFICIENT")
)

(gold_efficiency.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_trade_spend_efficiency"))

print(f"✅ gold_trade_spend_efficiency written: {gold_efficiency.count()} rows")
display(gold_efficiency.orderBy("avg_roi_pct"))

✅ gold_trade_spend_efficiency written: 7 rows


retailer_name,channel,promo_type,num_promotions,total_trade_spend_gbp,total_incremental_units,total_incremental_revenue,total_incremental_margin,avg_roi_pct,avg_volume_lift_pct,total_spend_per_incremental_unit,spend_efficiency_flag
DTC Ecommerce,DTC,Markdown,1,1000.0,-7557.0,-25965.74,-16641.67,-1764.2,-37.6,null,INEFFICIENT
Asda,Retail,Scan,2,9500.0,-7916.0,-15035.07,-4644.93,-149.05,-22.95,null,INEFFICIENT
Boots,Retail,Scan,4,13000.0,-9757.0,-12381.84,-4085.1800000000003,-137.075,-32.95,null,INEFFICIENT
Asda,Retail,Markdown,1,6000.0,-3126.0,-5086.89,-1304.16,-121.7,-80.5,null,INEFFICIENT
Amazon UK,DTC,Markdown,1,500.0,-96.0,-99.93,-2.06,-100.4,-6.2,null,INEFFICIENT
Smyths Toys,Retail,Markdown,1,4000.0,884.0,1804.21,652.94,-83.7,3.1,4.52,MODERATE
John Lewis,Retail,Feature,2,6000.0,35654.0,76164.6,36666.83,439.0,36.25,0.17,EFFICIENT


## Cell 11 — Gold: Promo ROI Ranking Scorecard

In [0]:
# Window for ranking
window_roi = Window.orderBy(F.col("promo_roi_pct").desc())

gold_ranking = df_gold_promos \
    .withColumn("roi_rank", F.rank().over(window_roi)) \
    .select(
        "roi_rank", "promo_id", "promo_name",
        "retailer_name", "channel", "promo_type",
        "disc_pct", "promo_duration_days",
        "trade_spend_gbp", "actual_gross_revenue",
        "incremental_units", "incremental_revenue_gbp",
        "incremental_margin_gbp", "net_incremental_margin_gbp",
        "promo_roi_pct", "volume_lift_pct",
        "roi_flag", "incrementality_flag"
    ).orderBy("roi_rank")

(gold_ranking.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_promo_roi_ranking"))

print(f"✅ gold_promo_roi_ranking written: {gold_ranking.count()} promotions ranked")
display(gold_ranking)

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ gold_promo_roi_ranking written: 12 promotions ranked


roi_rank,promo_id,promo_name,retailer_name,channel,promo_type,disc_pct,promo_duration_days,trade_spend_gbp,actual_gross_revenue,incremental_units,incremental_revenue_gbp,incremental_margin_gbp,net_incremental_margin_gbp,promo_roi_pct,volume_lift_pct,roi_flag,incrementality_flag
1,PRO-008,JL Christmas Nursery,John Lewis,Retail,Feature,15.0,29,3500.0,173150.3099999997,21934.0,65200.03,34005.8,30505.8,871.6,30.9,STRONG POSITIVE,HIGH LIFT
2,PRO-003,JL Spring Baby Fair,John Lewis,Retail,Feature,15.0,13,2500.0,61174.000000000146,13720.0,10964.57,2661.03,161.03,6.4,41.6,POSITIVE,HIGH LIFT
3,PRO-010,Boots Xmas Event,Boots,Retail,Scan,20.0,23,4000.0,22819.260000000115,-244.0,1566.65,885.74,-3114.26,-77.9,-1.8,VALUE DESTROYING,NO LIFT / CANNIBALISATION
4,PRO-005,Smyths Summer Sale,Smyths Toys,Retail,Markdown,30.0,29,4000.0,50980.77000000007,884.0,1804.21,652.94,-3347.06,-83.7,3.1,VALUE DESTROYING,LOW LIFT
5,PRO-012,Amazon Lightning Deal,Amazon UK,DTC,Markdown,30.0,0,500.0,2427.649999999999,-96.0,-99.93,-2.06,-502.06,-100.4,-6.2,VALUE DESTROYING,NO LIFT / CANNIBALISATION
6,PRO-007,Boots Baby Event Q3,Boots,Retail,Scan,20.0,13,3200.0,10528.240000000016,-1947.0,-1973.3,-582.04,-3782.04,-118.2,-23.8,VALUE DESTROYING,NO LIFT / CANNIBALISATION
7,PRO-009,Asda Black Friday Baby,Asda,Retail,Markdown,35.0,1,6000.0,1741.17,-3126.0,-5086.89,-1304.16,-7304.16,-121.7,-80.5,VALUE DESTROYING,NO LIFT / CANNIBALISATION
8,PRO-002,Asda Baby Week,Asda,Retail,Scan,25.0,6,5000.0,14826.920000000026,-4532.0,-7933.28,-2322.85,-7322.85,-146.5,-35.0,VALUE DESTROYING,NO LIFT / CANNIBALISATION
9,PRO-006,Asda Back to School,Asda,Retail,Scan,20.0,16,4500.0,47522.67999999993,-3384.0,-7101.79,-2322.08,-6822.08,-151.6,-10.9,VALUE DESTROYING,NO LIFT / CANNIBALISATION
10,PRO-001,Boots Baby Event,Boots,Retail,Scan,20.0,13,3000.0,6475.6599999999935,-4908.0,-6025.88,-1904.43,-4904.43,-163.5,-59.9,VALUE DESTROYING,NO LIFT / CANNIBALISATION


## Cell 12 — Module 2 Summary & Quality Check

In [0]:
print("=" * 65)
print("  MODULE 2 — QUALITY CHECK & TRADE SPEND SUMMARY")
print("=" * 65)

tables = [
    "gold_promo_events",
    "gold_trade_spend_efficiency",
    "gold_promo_roi_ranking",
]

for t in tables:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {DATABASE}.{t}").collect()[0]["n"]
    print(f"  ✅  {t:<35} {n:>6} rows")

print("=" * 65)

# Trade spend headline summary
summary = spark.table(f"{DATABASE}.gold_promo_events").agg(
    F.sum("trade_spend_gbp").alias("total_trade_spend"),
    F.sum("incremental_revenue_gbp").alias("total_incremental_revenue"),
    F.sum("incremental_margin_gbp").alias("total_incremental_margin"),
    F.sum("net_incremental_margin_gbp").alias("net_incremental_margin"),
    F.avg("promo_roi_pct").alias("avg_roi_pct"),
    F.avg("volume_lift_pct").alias("avg_volume_lift_pct"),
    F.count("promo_id").alias("total_promotions")
).collect()[0]

print()
print("  📊 TRADE SPEND HEADLINE NUMBERS")
print("  " + "-" * 50)
print(f"  Total Trade Spend          : £{summary['total_trade_spend']:>10,.0f}")
print(f"  Total Incremental Revenue  : £{summary['total_incremental_revenue']:>10,.0f}")
print(f"  Total Incremental Margin   : £{summary['total_incremental_margin']:>10,.0f}")
print(f"  Net Incremental Margin     : £{summary['net_incremental_margin']:>10,.0f}")
print(f"  Average Promo ROI          :  {summary['avg_roi_pct']:>10,.1f}%")
print(f"  Average Volume Lift        :  {summary['avg_volume_lift_pct']:>10,.1f}%")
print(f"  Total Promotions Analysed  :  {summary['total_promotions']:>10}")

print()
print("  🏆 ROI RANKING (Best → Worst)")
print("  " + "-" * 50)
spark.sql(f"""
    SELECT roi_rank, promo_name, retailer_name,
           promo_roi_pct, roi_flag
    FROM {DATABASE}.gold_promo_roi_ranking
    ORDER BY roi_rank
""").show(truncate=False)

print("=" * 65)
print("   MODULE 2 COMPLETE!")
print("  Next: Module 3 — Forecasting & Sell-Through")

  MODULE 2 — QUALITY CHECK & TRADE SPEND SUMMARY
  ✅  gold_promo_events                       12 rows
  ✅  gold_trade_spend_efficiency              7 rows
  ✅  gold_promo_roi_ranking                  12 rows

  📊 TRADE SPEND HEADLINE NUMBERS
  --------------------------------------------------
  Total Trade Spend          : £    40,000
  Total Incremental Revenue  : £    19,399
  Total Incremental Margin   : £    10,642
  Net Incremental Margin     : £   -29,358
  Average Promo ROI          :      -169.9%
  Average Volume Lift        :       -18.9%
  Total Promotions Analysed  :          12

  🏆 ROI RANKING (Best → Worst)
  --------------------------------------------------
+--------+----------------------+-------------+-------------+----------------+
|roi_rank|promo_name            |retailer_name|promo_roi_pct|roi_flag        |
+--------+----------------------+-------------+-------------+----------------+
|1       |JL Christmas Nursery  |John Lewis   |871.6        |STRONG POSITIVE |
|